In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


def main() -> None:
    spark = (
        SparkSession.builder
        .remote("sc://localhost:15002")
        .getOrCreate()
    )

    print(f"Spark version: {spark.version}")

    orders = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv("/opt/spark/data/orders.csv")
    )

    paid_orders = (
        orders
        .filter(F.col("status") == "paid")
        .withColumn(
            "amount_with_tax",
            F.round(F.col("amount") * F.lit(1.1), 2),
        )
    )

    summary = (
        paid_orders
        .groupBy("customer_id")
        .agg(
            F.count("*").alias("total_orders"),
            F.round(F.sum("amount"), 2).alias("total_amount"),
            F.round(F.sum("amount_with_tax"), 2).alias(
                "total_amount_with_tax"
            ),
        )
        .orderBy(F.desc("total_amount"))
    )

    print("Paid orders:")
    paid_orders.show(truncate=False)

    spark.stop()


if __name__ == "__main__":
    main()

Spark version: 4.1.2
Paid orders:
+--------+-----------+------+------+---------------+
|order_id|customer_id|amount|status|amount_with_tax|
+--------+-----------+------+------+---------------+
|1       |C001       |100.5 |paid  |110.55         |
|2       |C002       |250.0 |paid  |275.0          |
|4       |C003       |400.0 |paid  |440.0          |
|6       |C001       |320.0 |paid  |352.0          |
+--------+-----------+------+------+---------------+

